In [6]:
import pandas as pd
import os
import shutil

In [3]:
df = pd.read_csv("ADNI1_Complete_1Yr_1.5T_3_03_2026.csv")

df['Acq Date'] = pd.to_datetime(df['Acq Date'])

# ordenar por paciente + data
df_sorted = df.sort_values(by=['Subject', 'Acq Date'])

# pegar primeiro exame de cada paciente
df_unique = df_sorted.groupby('Subject').first().reset_index()

In [5]:
selected_ids = set(df_unique['Image Data ID'])

In [ ]:
import os
import shutil
from datetime import datetime
import csv

def fix_path(path):
    if not path.startswith("\\\\?\\"):
        return "\\\\?\\" + os.path.abspath(path)
    return path

source_root = fix_path(r"C:\Users\catri\OneDrive\Área de Trabalho\TCC\development_wokspace\Data\ADNI1_Complete 1Yr 1.5T\ADNI")
dest_root = fix_path(r"C:\Users\catri\OneDrive\Área de Trabalho\TCC\Data\ADNI_REDUZIDO")

os.makedirs(dest_root, exist_ok=True)

# 📊 métricas
stats = {
    "copied": 0,
    "failed": 0,
    "no_nii": 0,
    "filtered_out": 0
}

error_log = []

# 🧠 filtro MRI (remove PET e outros)
def is_mri(path):
    path = path.lower()
    return (
        "mpr" in path or
        "spgr" in path or
        "t1" in path
    ) and "pet" not in path


for subject in os.listdir(source_root):

    subject_path = os.path.join(source_root, subject)

    if not os.path.isdir(subject_path):
        continue

    try:
        nii_candidates = []

        # 🔎 busca TOTAL (não depende mais de data)
        for root, dirs, files in os.walk(subject_path):
            for f in files:
                if f.endswith(".nii") or f.endswith(".nii.gz"):

                    full_path = os.path.join(root, f)

                    if is_mri(full_path):
                        nii_candidates.append(full_path)
                    else:
                        stats["filtered_out"] += 1

        # ❌ nenhum arquivo
        if not nii_candidates:
            stats["no_nii"] += 1
            error_log.append([subject, "Nenhum NII encontrado"])
            print(f"⚠️ {subject} - nenhum .nii encontrado")
            continue

        # 🥇 pega o menor caminho (proxy de baseline)
        nii_candidates.sort(key=lambda x: len(x))
        selected_nii = nii_candidates[0]

        # 📁 destino
        dest_dir = os.path.join(dest_root, subject)
        os.makedirs(dest_dir, exist_ok=True)

        dest_file = os.path.join(dest_dir, os.path.basename(selected_nii))

        shutil.copy2(selected_nii, dest_file)

        stats["copied"] += 1
        print(f"✔ {subject}")

    except Exception as e:
        stats["failed"] += 1
        error_log.append([subject, str(e)])
        print(f"❌ erro {subject}: {e}")


# 💾 salva log CSV
log_file = os.path.join(dest_root, "relatorio_erros.csv")

with open(log_file, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Subject", "Erro"])
    writer.writerows(error_log)


# 📊 relatório final
print("\n===== RELATÓRIO FINAL =====")
for k, v in stats.items():
    print(f"{k}: {v}")

print(f"\n📄 Log salvo em: {log_file}")

✔ 002_S_0295
✔ 002_S_0413
✔ 002_S_0619
✔ 002_S_0685
✔ 002_S_0729
✔ 002_S_0782
✔ 002_S_0816
✔ 002_S_0938
✔ 002_S_0954
✔ 002_S_1018
✔ 002_S_1070
✔ 002_S_1155
✔ 002_S_1261
✔ 002_S_1268
✔ 002_S_1280
✔ 003_S_0907
✔ 003_S_0908
✔ 003_S_0981
✔ 003_S_1057
✔ 003_S_1122
✔ 005_S_0221
✔ 005_S_0222
✔ 005_S_0223
✔ 005_S_0324
✔ 005_S_0448
✔ 005_S_0546
✔ 005_S_0553
✔ 005_S_0602
✔ 005_S_0610
✔ 005_S_0814
✔ 005_S_1224
✔ 005_S_1341
✔ 006_S_0498
✔ 006_S_0547
✔ 006_S_0675
✔ 006_S_0681
✔ 006_S_0731
✔ 006_S_1130
✔ 007_S_0041
✔ 007_S_0068
✔ 007_S_0070
✔ 007_S_0101
✔ 007_S_0128
✔ 007_S_0249
✔ 007_S_0293
✔ 007_S_0316
✔ 007_S_0414
✔ 007_S_0698
✔ 007_S_1206
✔ 007_S_1222
✔ 007_S_1339
✔ 009_S_0751
✔ 009_S_0842
✔ 009_S_0862
✔ 009_S_1030
✔ 010_S_0067
✔ 010_S_0419
✔ 010_S_0422
✔ 010_S_0472
✔ 010_S_0786
✔ 010_S_0829
✔ 010_S_0904
✔ 011_S_0003
✔ 011_S_0005
✔ 011_S_0010
✔ 011_S_0016
✔ 011_S_0021
✔ 011_S_0022
✔ 011_S_0023
✔ 011_S_0053
✔ 011_S_0168
✔ 011_S_0183
✔ 011_S_0241
✔ 011_S_0326
✔ 011_S_0362
✔ 011_S_0856
✔ 011_S_0861